# Phase 6: Active Learning

## Step 6.1

In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

In [5]:
df = pd.read_csv('Train_Set_70_Augmented.csv')                                                                                                                                                                                                         
                                                                                                                                                                                                                                                        
# 2. Isolate the Authentic Pool                                                                                                                                                                                                                        
df_auth_pool = df[df['is_synthetic'] == False].copy()                                                                                                                                                                                                    
# 3. Extract the Initial Seed (Strict Stratification)                                                                                                                                                                                                  
# 20 samples * 5 rating categories = 100 total seed size                                                                                                                                                                                               
df_seed = df_auth_pool.groupby('rating').sample(n=20, random_state=42)                                                                                                                                                                     
df_unannotated_pool = df_auth_pool.drop(df_seed.index)                                                                                                                                                                                                                                                                                                                                                                             
df_seed.to_csv('AL_Seed_Batch_1.csv', index=False, encoding='utf-8')                                                                                                                                                                                   
                                                                        

In [6]:
df_seed.to_csv('AL_Seed_Batch_1.csv', index=False, encoding='utf-8')                                                                                                                                                                                   
                                                                                                                                                                                                                                                        
print(f"Total Authentic Pool Available: {len(df_auth_pool)}")                                                                                                                                                                                          
print(f"Extracted AL Seed Size: {len(df_seed)}")                                                                                                                                                                                                       
                                                                                                                                                                                                                                                        
print("\nSeed Distribution by Star Rating:")                                                                                                                                                                                                           
print(df_seed['rating'].value_counts().sort_index())                                                                                                                                                                                                   
                                                                                                                                                                                                                                                        
print(f"\nRemaining Unannotated Pool: {len(df_unannotated_pool)}")    

Total Authentic Pool Available: 16140
Extracted AL Seed Size: 100

Seed Distribution by Star Rating:
rating
1    20
2    20
3    20
4    20
5    20
Name: count, dtype: int64

Remaining Unannotated Pool: 16040


# Fleiss Kappa Testing

In [5]:
base_dir = r"D:\Ateneo de Davao\Thesis-Project-Aspect-Based-Sentiment-Analysis\_6_Active_Learning_w_Human_Annotation"
f1 = pd.read_csv(os.path.join(base_dir, "Copy 1 Seed_Batch.csv"))
f2 = pd.read_csv(os.path.join(base_dir, "Copy 2 Seed_Batch.csv"))
f3 = pd.read_csv(os.path.join(base_dir, "Copy 3 Seed_Batch.csv"))
f4 = pd.read_csv(os.path.join(base_dir, "Copy 4 Seed_Batch.csv"))

In [6]:
ratings_df = pd.DataFrame({
    'rater1': f1['sentiment'].astype(str).str.strip().str.title(),
    'rater2': f2['sentiment'].astype(str).str.strip().str.title(),
    'rater3': f3['sentiment'].astype(str).str.strip().str.title(),
    'rater4': f4['sentiment'].astype(str).str.strip().str.title()
})

In [7]:
ratings_df = ratings_df.replace('Nan', np.nan).dropna()
print(f"Total fully annotated sentences: {len(ratings_df)}")

Total fully annotated sentences: 100


In [8]:
# Build the N x k matrix counting the votes per sentence
ratings_matrix = []

for index, row in ratings_df.iterrows():
    counts = [
        (row == 'Positive').sum(),
        (row == 'Neutral').sum(),
        (row == 'Negative').sum()
    ]
    ratings_matrix.append(counts)

ratings_matrix = np.array(ratings_matrix)
print("Preview of the generated matrix (first 5 rows):")
print(ratings_matrix[:5])

Preview of the generated matrix (first 5 rows):
[[0 1 3]
 [0 0 4]
 [0 0 4]
 [0 0 4]
 [0 0 4]]


In [9]:
def fleiss_kappa(matrix):
    """
    Computes Fleiss' Kappa for an item-category rater matrix.
    """
    X = np.array(matrix, dtype=float)
    N, k = X.shape

    n = np.sum(X[0])

    # 1. Calculate p_j (chance agreement components)
    p_j = np.sum(X, axis=0) / (N * n)

    # 2. Calculate P_e_bar (expected chance agreement)
    P_e_bar = np.sum(p_j ** 2)

    # 3. Calculate P_i for each subject
    P_i = (np.sum(X ** 2, axis=1) - n) / (n * (n - 1))

    # 4. Calculate P_bar (mean observed agreement)
    P_bar = np.mean(P_i)

    # 5. Calculate Fleiss' Kappa (κ)
    # Prevent division by zero if there's perfect expected agreement
    if P_e_bar == 1.0:
        return 1.0

    kappa = (P_bar - P_e_bar) / (1 - P_e_bar)

    return kappa

In [10]:
kappa_score = fleiss_kappa(ratings_matrix)
print(f"Final Fleiss' Kappa (κ): {kappa_score:.4f}")

# Standard Interpretation Guidelines:
if kappa_score < 0:
    print("Interpretation: Poor agreement")
elif kappa_score <= 0.20:
    print("Interpretation: Slight agreement")
elif kappa_score <= 0.40:
    print("Interpretation: Fair agreement")
elif kappa_score <= 0.60:
    print("Interpretation: Moderate agreement")
elif kappa_score <= 0.80:
    print("Interpretation: Substantial agreement")
else:
    print("Interpretation: Almost perfect agreement")


Final Fleiss' Kappa (κ): 0.6550
Interpretation: Substantial agreement


K > 0.61, Proceed

#Step 6.2: Uncertainty and Diversity Sampling

In [13]:
from collections import Counter

In [14]:
#re-annotation for sentences with ties in votes
df_base = pd.read_csv('AL_Seed_Batch_1.csv')                                                                         
                                                                                                                    
f1 = pd.read_csv('Copy 1 Seed_Batch.csv')                                                                            
f2 = pd.read_csv('Copy 2 Seed_Batch.csv')                                                                            
f3 = pd.read_csv('Copy 3 Seed_Batch.csv')                                                                            
f4 = pd.read_csv('Copy 4 Seed_Batch.csv')  

#-------

df_base['annotator_1'] = f1['sentiment'].str.strip().str.upper()                                                     
df_base['annotator_2'] = f2['sentiment'].str.strip().str.upper()                                                     
df_base['annotator_3'] = f3['sentiment'].str.strip().str.upper()                                                     
df_base['annotator_4'] = f4['sentiment'].str.strip().str.upper()                                                     
                                                                    

In [16]:
annotator_cols = ['annotator_1', 'annotator_2', 'annotator_3', 'annotator_4']                                        
                                                                                                                        
def resolve_majority(row):                                                                                           
    votes = [row[c] for c in annotator_cols if pd.notna(row[c])]                                                     
    counts = Counter(votes).most_common()                                                                            
    breakdown = ', '.join([f"{label}:{count}" for label, count in counts])                                           
                                                                                                                        
    top_label, top_count = counts[0]                                                                                 
                                                                                                                        
    # 2-2 tie or 2-1-1 split                                                                                         
    if (len(counts) > 1 and top_count == counts[1][1]) or (top_count == 2 and len(counts) == 3):                     
        return pd.Series({'ground_truth': 'TIE_NEEDS_REVIEW', 'vote_breakdown': breakdown, 'is_tie': True})          
                                                                                                                        
    # Majority (3-1, 4-0) or 3-vote dominance                                                                        
    return pd.Series({'ground_truth': top_label, 'vote_breakdown': breakdown, 'is_tie': False})                      
                                                                                                                        
results = df_base.apply(resolve_majority, axis=1)                                                                    
df_consolidated = pd.concat([df_base, results], axis=1)                                                              
                                                                                                                        
# Print Summary Metrics                                                                                           
ties = df_consolidated[df_consolidated['is_tie']]                                                                    
print(f"Total evaluated rows: {len(df_consolidated)}")                                                               
print(f"Consensus agreed (>=3 votes): {len(df_consolidated) - len(ties)}")                                           
print(f"Ties requiring manual tie-breaker: {len(ties)}")                                                             
                                                                                                                        
                                                
if len(ties) > 0:                                                                                                    
    print("\n--- Rows Requiring Manual Adjustment ---")                                                            
    display(ties[['sentence', 'TargetAspect', 'vote_breakdown', 'annotator_1', 'annotator_2', 'annotator_3',         
'annotator_4']])                                                                                                       
                                                                                                                        
# 4. Save to Seed_Ground_Truth.csv                                                                                   
df_consolidated.to_csv('Seed_Ground_Truth.csv', index=False, encoding='utf-8')                                       
print("\nSaved output to 'Seed_Ground_Truth.csv'")   

Total evaluated rows: 100
Consensus agreed (>=3 votes): 92
Ties requiring manual tie-breaker: 8

--- Rows Requiring Manual Adjustment ---


,sentence,TargetAspect,vote_breakdown,annotator_1,annotator_2,annotator_3,annotator_4
6,"Thirty minutes before 11 PM, our drinks were s...",Product Quality,"NEGATIVE:2, NEUTRAL:2",NEGATIVE,NEGATIVE,NEUTRAL,NEUTRAL
28,The place is a little bit crowded which is und...,Ambiance and Atmosphere,"NEUTRAL:2, POSITIVE:2",NEUTRAL,POSITIVE,NEUTRAL,POSITIVE
36,Still on the menu are the old faves — Chicken ...,Product Quality,"NEGATIVE:2, POSITIVE:1, NEUTRAL:1",POSITIVE,NEGATIVE,NEUTRAL,NEGATIVE
41,"The staff is exceptionally polite; however, th...",Customer Service,"POSITIVE:2, NEGATIVE:2",POSITIVE,NEGATIVE,POSITIVE,NEGATIVE
54,"We also paired it with some of their cookies, ...",Product Quality,"NEGATIVE:2, NEUTRAL:2",NEGATIVE,NEGATIVE,NEUTRAL,NEUTRAL
58,Magazines on the shelf for time spending on th...,Product Quality,"NEUTRAL:2, POSITIVE:2",NEUTRAL,POSITIVE,NEUTRAL,POSITIVE
59,"Honestly, I thought you bought your cakes from...",Product Quality,"POSITIVE:2, NEGATIVE:1, NEUTRAL:1",NEGATIVE,POSITIVE,NEUTRAL,POSITIVE
60,2023 update: Main reason for coming here is to...,Product Quality,"NEUTRAL:2, NEGATIVE:1, POSITIVE:1",NEGATIVE,NEUTRAL,NEUTRAL,POSITIVE



Saved output to 'Seed_Ground_Truth.csv'


In [17]:
ties_df = df_consolidated[df_consolidated['is_tie']].copy()                                                                                                                       
ties_df['final_sentiment'] = ''                                                                                      
                                                                                                                    
                                                                          
review_cols = [                                                                                                      
'sentence',                                                                                                      
'TargetAspect',                                                                                                  
'rating',                                                                                                        
'vote_breakdown',                                                                                                
'annotator_1',                                                                                                   
'annotator_2',                                                                                                   
'annotator_3',                                                                                                   
'annotator_4',                                                                                                   
'final_sentiment'                                                                                                
]                                                                                                                    
                                                                                                                    
                                                                                            
ties_df[review_cols].to_csv('Ties_For_Manual_Review.csv', index=False, encoding='utf-8')                             
print(f"Successfully exported {len(ties_df)} tied rows to 'Ties_For_Manual_Review.csv'")   

Successfully exported 8 tied rows to 'Ties_For_Manual_Review.csv'


In [18]:
df_gt = pd.read_csv('Seed_Ground_Truth.csv')                                                                         
df_ties = pd.read_csv('Ties_For_Manual_Review.csv')                                                                  
                                                                  
tie_map = dict(zip(df_ties['sentence'], df_ties['final_sentiment'].str.strip().str.upper()))                                        
for sentence, resolved_sentiment in tie_map.items():                                                                 
    mask = df_gt['sentence'] == sentence                                                                             
    df_gt.loc[mask, 'ground_truth'] = resolved_sentiment                                                             
df_gt['sentiment'] = df_gt['ground_truth']                                                                                                                             
remaining_unresolved = (df_gt['ground_truth'] == 'TIE_NEEDS_REVIEW').sum()                                           
print(f"Remaining unresolved ties: {remaining_unresolved}")                                                                                                                                                                   
print("\nFinal Seed Ground Truth Label Distribution:")                                                               
print(df_gt['sentiment'].value_counts())                                                                             

df_gt.to_csv('Seed_Ground_Truth.csv', index=False, encoding='utf-8')                                                 
print("\nSuccessfully updated 'Seed_Ground_Truth.csv'.")        

Remaining unresolved ties: 0

Final Seed Ground Truth Label Distribution:
sentiment
NEGATIVE    42
POSITIVE    39
NEUTRAL     19
Name: count, dtype: int64

Successfully updated 'Seed_Ground_Truth.csv'.


In [21]:
import torch
from torch.utils.data import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments, set_seed

d:\Ateneo de Davao\Thesis-Project-Aspect-Based-Sentiment-Analysis\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [22]:
set_seed(42)                                                                                                         
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')                                                
print(f"Using device: {device}") 

Using device: cpu


In [23]:
df_seed = pd.read_csv('Seed_Ground_Truth.csv')                                                                       
                                                                                                                        
                                                                            
label2id = {'NEGATIVE': 0, 'NEUTRAL': 1, 'POSITIVE': 2}                                                              
id2label = {v: k for k, v in label2id.items()}                                                                       
                                                                                                                        
df_seed['label'] = df_seed['sentiment'].str.strip().str.upper().map(label2id)                                                                                                                 
df_seed = df_seed.dropna(subset=['label', 'sentence', 'TargetAspect']).copy()                                        
df_seed['label'] = df_seed['label'].astype(int)                                                                      
                                                                                                                        
# Initialize XLM-RoBERTa Tokenizer                                                                                
model_name = 'xlm-roberta-base'                                                                                      
tokenizer = AutoTokenizer.from_pretrained(model_name)    

d:\Ateneo de Davao\Thesis-Project-Aspect-Based-Sentiment-Analysis\.venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\loren\.cache\huggingface\hub\models--xlm-roberta-base. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


In [24]:
class ABSADataset(Dataset):                                                                                          
    def __init__(self, df, tokenizer, max_len=128, is_test=False):                                                   
        self.texts_aspect = df['TargetAspect'].astype(str).tolist()                                                  
        self.texts_sentence = df['sentence'].astype(str).tolist()                                                    
        self.tokenizer = tokenizer                                                                                   
        self.max_len = max_len                                                                                       
        self.is_test = is_test                                                                                       
        if not is_test:                                                                                              
            self.labels = df['label'].tolist()                                                                       
                                                                                                                        
    def __len__(self):                                                                                               
        return len(self.texts_sentence)                                                                              
                                                                                                                        
    def __getitem__(self, idx):                                                                                      
        # Sequence-pair encoding: [CLS] TargetAspect [SEP] Sentence [SEP]                                            
        encoding = self.tokenizer(                                                                                   
            self.texts_aspect[idx],                                                                                  
            self.texts_sentence[idx],                                                                                
            truncation=True,                                                                                         
            padding='max_length',                                                                                    
            max_length=self.max_len,                                                                                 
            return_tensors='pt'                                                                                      
        )                                                                                                            
        item = {key: val.squeeze(0) for key, val in encoding.items()}                                                
        if not self.is_test:                                                                                         
            item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)                                        
        return item                                                                                                  
                                                                                                                        
train_dataset = ABSADataset(df_seed, tokenizer)                                                                      
print(f"Loaded {len(train_dataset)} seed instances for initial baseline fine-tuning.")  

Loaded 100 seed instances for initial baseline fine-tuning.


In [26]:
from torch.utils.data import DataLoader                                                                              
from transformers import AutoModelForSequenceClassification, get_linear_schedule_with_warmup                         
from torch.optim import AdamW                                                                                        
                                                                                                                        
                                                                                    
model = AutoModelForSequenceClassification.from_pretrained(                                                          
    model_name,                                                                                                      
    num_labels=3,                                                                                                    
    id2label=id2label,                                                                                               
    label2id=label2id                                                                                                
).to(device)                                                                                                         
                                                                                                                        
# Setup DataLoader & Optimizer                                                                                    
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)                                                 
optimizer = AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)                                                    
                                                                                                                        
epochs = 5                                                                                                           
total_steps = len(train_loader) * epochs                                                                             
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=0, num_training_steps=total_steps)           
                                                                                                                        
#p                                                                                     
print("Starting Native Baseline Training on Seed Ground Truth...")                                                   
model.train()                                                                                                        
                                                                                                                        
for epoch in range(epochs):                                                                                          
    total_loss = 0                                                                                                   
    for batch in train_loader:                                                                                       
        optimizer.zero_grad()                                                                                        
                                                                                                                        
        input_ids = batch['input_ids'].to(device)                                                                    
        attention_mask = batch['attention_mask'].to(device)                                                          
        labels = batch['labels'].to(device)                                                                          
                                                                                                                        
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)                           
        loss = outputs.loss                                                                                          
                                                                                                                        
        loss.backward()                                                                                              
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)                                             
                                                                                                                        
        optimizer.step()                                                                                             
        scheduler.step()                                                                                             
                                                                                                                        
        total_loss += loss.item()                                                                                    
                                                                                                                        
    avg_loss = total_loss / len(train_loader)                                                                        
    print(f"Epoch {epoch + 1}/{epochs} - Average Loss: {avg_loss:.4f}")                                              
                                                                                                                        
print("Baseline Engine Training Complete.")     

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 525.42it/s]
[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
classifier.out_proj.bias    | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.dense.bias       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Starting Native Baseline Training on Seed Ground Truth...
Epoch 1/5 - Average Loss: 1.1090
Epoch 2/5 - Average Loss: 1.0816
Epoch 3/5 - Average Loss: 1.0733
Epoch 4/5 - Average Loss: 1.0636
Epoch 5/5 - Average Loss: 1.0541
Baseline Engine Training Complete.


In [27]:
from torch.utils.data import DataLoader                                                                              
from tqdm.auto import tqdm                                                                                           
                                                                                                                        
# 1. Load Augmented Dataset & Isolate the Unannotated Pool                                                           
df_all = pd.read_csv('Train_Set_70_Augmented.csv')                                                                   
                                                                                                                        
# Exclude sentences already annotated in Seed_Ground_Truth.csv                                                       
annotated_sentences = set(df_seed['sentence'].astype(str))                                                           
df_unannotated = df_all[~df_all['sentence'].astype(str).isin(annotated_sentences)].copy().reset_index(drop=True)     
                                                                                                                        
print(f"Unannotated Pool Size: {len(df_unannotated)} rows")                                                          
                                                                                                                        
# 2. Setup Unannotated DataLoader                                                                                    
unannotated_dataset = ABSADataset(df_unannotated, tokenizer, is_test=True)                                           
unannotated_loader = DataLoader(unannotated_dataset, batch_size=32, shuffle=False)                                   
                                                                                                                        
# 3. Run Inference & Extract Softmax Probabilities                                                                   
model.eval()                                                                                                         
all_probs = []                                                                                                       
                                                                                                                        
with torch.no_grad():                                                                                                
    for batch in tqdm(unannotated_loader, desc="Calculating Probabilities"):                                         
        input_ids = batch['input_ids'].to(device)                                                                    
        attention_mask = batch['attention_mask'].to(device)                                                          
                                                                                                                        
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)                                          
        probs = torch.softmax(outputs.logits, dim=1).cpu().numpy()                                                   
        all_probs.append(probs)                                                                                      
                                                                                                                        
all_probs = np.vstack(all_probs)                                                                                     
                                                                                                                        
# 4. Compute Margin (Difference between Top 1 and Top 2 Probabilities)                                               
# Sort probabilities for each row ascending -> last two elements are top2 and top1                                   
sorted_probs = np.sort(all_probs, axis=1)                                                                            
top1_prob = sorted_probs[:, -1]                                                                                      
top2_prob = sorted_probs[:, -2]                                                                                      
                                                                                                                        
df_unannotated['prob_negative'] = all_probs[:, 0]                                                                    
df_unannotated['prob_neutral'] = all_probs[:, 1]                                                                     
df_unannotated['prob_positive'] = all_probs[:, 2]                                                                    
df_unannotated['predicted_label'] = np.argmax(all_probs, axis=1)                                                     
df_unannotated['predicted_label'] = df_unannotated['predicted_label'].map(id2label)                                  
df_unannotated['margin'] = top1_prob - top2_prob                                                                     
                                                                                                                        
# 5. Sort from Lowest Margin (Highest Uncertainty / Most Ambiguous) to Highest                                       
df_ranked = df_unannotated.sort_values(by='margin', ascending=True).reset_index(drop=True)                           
                                                                                                                        
# 6. Save Ranked Dataset                                                                                             
output_file = 'Unannotated_Pool_Ranked_Margin.csv'                                                                   
df_ranked.to_csv(output_file, index=False, encoding='utf-8')                                                         
print(f"\nSaved ranked uncertainty pool to '{output_file}'")                                                         
                                                                                                                        
# 7. Preview Top-10 Most Ambiguous Candidates (AL Batch 2 Selection)                                                 
display_cols = ['sentence', 'TargetAspect', 'predicted_label', 'margin', 'prob_negative', 'prob_neutral',            
'prob_positive']                                                                                                       
print("\n--- Top 10 Most Ambiguous Samples (Smallest Margin) ---")                                                   
display(df_ranked[display_cols].head(10))  

Unannotated Pool Size: 16482 rows


Calculating Probabilities:  28%|██▊       | 147/516 [05:26<13:38,  2.22s/it]


KeyboardInterrupt: 